In [4]:
import asyncio
import json
from pathlib import Path
from typing import Dict, Any, Tuple, Set

import polars as pl
from tqdm import tqdm
from openai import AsyncOpenAI


CATEGORY = "Pet_Supplies"
DATA_DIR = Path.cwd()
INPUT_PATH = f"prepared/{CATEGORY}_items_cleaned.parquet"
OUTPUT_PATH = f"prepared/{CATEGORY}_extracted.parquet"
CHECKPOINT_PATH = Path("checkpoint/extracted_data.jsonl")

OPENROUTER_API_KEY = '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'
MODEL = "google/gemini-2.5-flash-lite"
CONCURRENCY = 50

client = AsyncOpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

In [8]:
SYSTEM_PROMPT = """
You extract compact semantic metadata for pet products.

Return STRICTLY valid JSON with EXACTLY these fields:
{
  "product_type": string|null,
  "pet_species": string|null,
  "life_stage": string|null,
  "primary_use_case": string|null,
  "material_or_active": [string,...]|null
}

Rules:
- Use ONLY information explicitly present in the input. Do NOT guess or infer.
- Normalize values:
  - pet_species: one of ["dog","cat","fish","bird","small_animal","reptile","horse","other"].
  - life_stage: one of ["puppy","kitten","adult","senior","all"].
  - product_type: short concrete noun such as:
    harness, leash, collar, toy, treat, dry_food, wet_food, supplement,
    bed, cave_bed, carrier, litter, litter_box, scratching_post, cat_tree,
    filter, heater, lamp, gravel, bowl, feeder, fountain, cage, hutch, coat, sweater, dress, boots.
  - primary_use_case: one of:
    walk, train, chew, play, sleep, scratch, joint_support, grooming,
    filtration, lighting, heating, feeding, hydration, travel, litter_management.
  - material_or_active: lowercase snake_case strings; deduplicate.
- Return null for any field you cannot confidently extract.
- For list fields: return null (not empty list) if nothing is extracted.
- No extra text. Return JSON only.
""".strip()

In [9]:
# -----------------------------
# Context builder (compact + robust)
# -----------------------------
def build_item_context(row: dict) -> str:
    def g(k: str) -> str:
        v = row.get(k)
        return "" if v is None else str(v)

    title = g("clean_title")
    desc = g("clean_description")
    cat = g("categories_text") or g("main_category")
    features = g("clean_features")

    parts = []
    if title:
        parts.append(f"Title: {title}")
    if desc:
        parts.append(f"Description: {desc}")
    if features:
        parts.append(f"Features: {features}")
    if cat:
        parts.append(f"Category: {cat}")

    return "\n".join(parts).strip()


# -----------------------------
# Checkpoint helpers (single pass)
# -----------------------------
def load_checkpoint_index(
    path: Path,
    id_key: str = "parent_asin",
) -> Tuple[Set[str], Dict[str, dict]]:
    """
    Один проход по JSONL чекпоинту:
    - processed_success: asin, которые успешно обработаны (error == None)
    - latest_by_asin: последняя запись по asin (для merge при рестарте)
    """
    processed_success: Set[str] = set()
    latest_by_asin: Dict[str, dict] = {}

    if not path.exists():
        return processed_success, latest_by_asin

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue

            asin = obj.get(id_key)
            if not asin:
                continue

            latest_by_asin[asin] = obj
            if obj.get("error") is None:
                processed_success.add(asin)

    return processed_success, latest_by_asin


def append_jsonl(path: Path, obj: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def strip_code_fences(s: str) -> str:
    s = (s or "").strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[1]
        if s.endswith("```"):
            s = s[:-3]
    return s.strip()


# -----------------------------
# API call per item
# -----------------------------
async def extract_one(parent_asin: str, item_context: str) -> dict:
    try:
        resp = await client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": item_context},
            ],
            temperature=0,
        )

        content = strip_code_fences(resp.choices[0].message.content or "")
        data = json.loads(content)

        # минимальная нормализация формы
        return {
            "parent_asin": parent_asin,
            "product_type": data.get("product_type"),
            "pet_species": data.get("pet_species"),
            "life_stage": data.get("life_stage"),
            "primary_use_case": data.get("primary_use_case"),
            "material_or_active": data.get("material_or_active"),
            "error": None,
        }

    except Exception as e:
        return {
            "parent_asin": parent_asin,
            "product_type": None,
            "pet_species": None,
            "life_stage": None,
            "primary_use_case": None,
            "material_or_active": None,
            "error": f"{type(e).__name__}: {e}",
        }


# -----------------------------
# Main Polars pipeline (async + progress)
# -----------------------------
async def extract_pet_metadata_polars(
    df: pl.DataFrame,
    checkpoint_path: str | Path,
    id_col: str = "parent_asin",
    concurrency: int = 16,
) -> pl.DataFrame:
    concurrency = max(1, int(concurrency))
    checkpoint_path = Path(checkpoint_path)

    # 1) чекпоинт: и skip, и merge — одним проходом
    processed_success, latest_map = load_checkpoint_index(checkpoint_path, id_key=id_col)

    # 2) todo: ретраим фейлы (error != None) — они НЕ попадают в processed_success
    todo = df.filter(~pl.col(id_col).is_in(list(processed_success)))
    total = todo.height

    # 3) семафор для параллелизма
    sem = asyncio.Semaphore(concurrency)

    async def run_row(row: dict) -> dict:
        async with sem:
            item_context = build_item_context(row)
            return await extract_one(row[id_col], item_context)

    new_results: list[dict] = []
    tasks: list[asyncio.Task] = []

    with tqdm(total=total, desc="Extracting pet metadata") as pbar:
        for row in todo.iter_rows(named=True):
            tasks.append(asyncio.create_task(run_row(row)))

            if len(tasks) >= concurrency:
                done = await asyncio.gather(*tasks)
                for r in done:
                    append_jsonl(checkpoint_path, r)
                    new_results.append(r)
                pbar.update(len(done))
                tasks.clear()

        if tasks:
            done = await asyncio.gather(*tasks)
            for r in done:
                append_jsonl(checkpoint_path, r)
                new_results.append(r)
            pbar.update(len(done))

    # 4) merge: latest_map (из чекпоинта) + новые результаты (последние побеждают)
    for r in new_results:
        asin = r.get(id_col)
        if asin:
            latest_map[asin] = r

    # 5) join back
    if latest_map:
        # Normalize records: material_or_active must be list or None
        # (old checkpoint entries may have it as a string due to LLM returning wrong type)
        records = list(latest_map.values())
        for r in records:
            mat = r.get("material_or_active")
            if mat is not None and not isinstance(mat, list):
                r["material_or_active"] = None

        res_df = pl.DataFrame(records, infer_schema_length=len(records))

        wanted_cols = [
            id_col,
            "product_type",
            "pet_species",
            "life_stage",
            "primary_use_case",
            "material_or_active",
        ]
        for c in wanted_cols:
            if c not in res_df.columns:
                res_df = res_df.with_columns(pl.lit(None).alias(c))
        res_df = res_df.select(wanted_cols)

        return df.join(res_df, on=id_col, how="left")

    # no checkpoint + no results
    return df.with_columns([
        pl.lit(None).alias("product_type"),
        pl.lit(None).alias("pet_species"),
        pl.lit(None).alias("life_stage"),
        pl.lit(None).alias("primary_use_case"),
        pl.lit(None).alias("material_or_active"),
    ])

In [10]:
df = pl.read_parquet(INPUT_PATH)
out = await extract_pet_metadata_polars(df, CHECKPOINT_PATH, concurrency=CONCURRENCY)
out.write_parquet(DATA_DIR / "prepared" / f"{CATEGORY}_items_with_metadata.parquet")

Extracting pet metadata: 100%|██████████| 1/1 [00:01<00:00,  1.34s/it]


In [11]:
out

parent_asin,main_category,title,average_rating,rating_number,features_text,description_text,price,store,categories_text,item_context,clean_title,clean_description,clean_features,product_type,pet_species,life_stage,primary_use_case,material_or_active
str,str,str,f32,i32,str,str,f32,str,str,str,str,str,list[str],str,str,str,str,list[str]
"""B00XJG2SLG""","""Pet Supplies""","""Hurtta Pet Collection 14-Inch …",4.4,166,"""Made from highly durable Neopr…","""Hurtta harnesses are suitable …",24.950001,"""Hurtta""","""Pet Supplies > Dogs > Collars,…","""Product: Hurtta Pet Collection…","""Hurtta Pet Collection 14-Inch …","""Hurtta harnesses are suitable …","[""Made from durable Neoprene"", ""Fitted with efficient 3M reflectors"", … ""Protection from the clip buckle""]","""harness""","""dog""","""all""","""walk""","[""neoprene"", ""3m reflectors""]"
"""B01MQTWB5H""","""Pet Supplies""","""4 Pack - 4 Inch Ring Filter So…",4.4,84,"""Micron filter bags provide exc…","""Micron filter bags provide exc…",15.0,"""Encompass All""","""""","""Product: 4 Pack - 4 Inch Ring …","""4 Pack - 4 Inch Ring Filter So…","""These 4-inch ring filter socks…","[""4 pack of 4-inch ring filter socks"", ""200-micron filtration"", … ""Durable and reusable""]","""filter""","""fish""",null,"""filtration""","[""felt""]"
"""B00F3JRLYQ""","""Pet Supplies""","""Wysong Optimal Adult Canine Fo…",4.4,75,"""42% Protein With High Levels O…","""For the past 35 years the Wyso…",17.459999,"""Wysong""","""Pet Supplies > Dogs > Food > D…","""Product: Wysong Optimal Adult …","""Wysong Optimal Adult Canine Fo…","""Wysong Optimal Adult super pre…","[""42% protein with high levels of fresh/frozen and dried meats"", ""Natural protein and fat"", … ""Wysong has been a leader in pet nutrition since 1979""]","""dry_food""","""dog""","""all""","""feeding""","[""probiotics"", ""enzymes"", … ""omega-6""]"
"""B09J5FJYMJ""","""Pet Supplies""","""Nutramax Dasuquin Joint Health…",4.3,641,"""Joint Health Support for Cats:…","""Dasuquin for Cats Soft Chews i…",9.68,"""Dasuquin""","""Pet Supplies > Cats > Health S…","""Product: Nutramax Dasuquin Joi…","""Nutramax Dasuquin Joint Health…","""Dasuquin for Cats Soft Chews i…","[""Joint health support for cats"", ""Contains ASU, glucosamine, and chondroitin sulfate"", … ""Veterinarian formulated with high-quality ingredients""]","""supplement""","""cat""","""all""","""joint_support""","[""glucosamine"", ""chondroitin"", … ""omega-3""]"
"""B08WZ1Z9ZF""","""Pet Supplies""","""Batiyeer 4 Strings in 4 Pieces…",4.0,95,"""Reliable to use: due to the qu…","""Features: Remind other animal…",null,"""Batiyeer""","""Pet Supplies > Dogs > Collars,…","""Product: Batiyeer 4 Strings in…","""Batiyeer 4 Strings in 4 Pieces…","""These collar bells are made of…","[""Made of quality brass material."", ""Durable, tough, and not easy to break or corrode."", … ""Enough for daily use and replacement.""]","""collar_charm""","""dog""","""all""","""walk""","[""brass""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""B0013ZEGWO""","""Pet Supplies""","""Sera 7112 Koi Royal Mini 1.8 l…",4.6,1054,"""Staple food for Koi Strengthen…","""Sera Koi Royal is the staple f…",20.75,"""Sera""","""Pet Supplies > Fish & Aquatic …","""Product: Sera 7112 Koi Royal M…","""Sera 7112 Koi Royal Mini 1.8 l…","""Sera Koi Royal is a staple foo…","[""Staple food for Koi"", ""Strengthens the immune system"", … ""For Koi up to 5 inches""]","""dry_food""","""fish""","""all""","""feeding""","[""wheat germ"", ""omega fatty acids"", ""mannan oligosaccharides""]"
"""B099J628C8""","""""","""Guess What? My Mom is Pregnant…",4.6,161,"""Texture of material：We use 100…","""Trendy Designs: Our goal was t…",10.99,"""Yangmics Direct""","""Pet Supplies > Dogs > Apparel …","""Product: Guess What? My Mom is…","""Pregnancy Announcement Dog Ban…","""These trendy, neutral bandanas…","[""Made from 100% soft spun polyester"", ""Breathable material keeps dogs cool"", … ""Pre-folded and easy to wear""]","""bandana""","""dog""","""all""